# 🏗️ Ingesting & Transforming an Engineering Manual into a Knowledge Graph with `AI.CHUNK_DOCUMENT` & Gemini

This notebook demonstrates how an unstructured engineering manual (**`manual_frozone_zenith_5000.pdf`** — the FroZone Zenith 5000 commercial HVAC manual) stored in Google Cloud Storage is semantically chunked in BigQuery using **[`AI.CHUNK_DOCUMENT`](https://docs.cloud.google.com/bigquery/docs/reference/standard-sql/bigqueryml-syntax-ai-chunk-document)** and transformed into structured **Knowledge Graph Triples** using **Gemini 3.8 Flash** (`AI.GENERATE`).

### 📊 Notebook Workflow & Architecture Overview
```mermaid
flowchart TD
    subgraph GCS ["1. Google Cloud Storage (Unstructured Data)"]
        PDF["manual_frozone_zenith_5000.pdf<br/>(2-Page Engineering Manual PDF)"]
    end

    subgraph BQ_CONN ["2. Cloud Storage URI or External Object Table"]
        CONN["EUC or Cloud Resource Connection<br/>(__default_cloudresource_connection__)"]
        OBJ_TBL["GCS URI Pattern / Object Table<br/>(kg_demo.frozone_manual_object_table)"]
    end

    subgraph DOC_AI ["3. Managed Layout Chunking (AI.CHUNK_DOCUMENT)"]
        PARSE["SQL: AI.CHUNK_DOCUMENT(..., chunk_size => 250)<br/>(Auto-manages bq-created-layout-parser)"]
        CHUNKS["Semantic Document Chunks Table<br/>(kg_demo.frozone_parsed_chunks)"]
    end

    subgraph GEMINI ["4. Triple Extraction (AI.GENERATE - Gemini 3.8 Flash)"]
        EXTRACT["SQL: AI.GENERATE(..., endpoint => 'gemini-3.8-flash')<br/>Extract Triples & Entity Types"]
        TRIPLES["Extracted Triples Table<br/>(kg_demo.frozone_extracted_triples)"]
    end

    subgraph KG_MERGE ["5. Knowledge Graph Integration & SQL/GQL"]
        MERGE["Merge into Enterprise Graph<br/>(kg_demo.extracted_knowledge_graph)"]
        QUERY["Multi-Hop BOM Tree Query<br/>(Product -> Part -> Material)"]
    end

    PDF --> CONN
    CONN --> OBJ_TBL
    OBJ_TBL --> PARSE
    PARSE --> CHUNKS
    CHUNKS --> EXTRACT
    EXTRACT --> TRIPLES
    TRIPLES --> MERGE
    MERGE --> QUERY
```

> [!NOTE]
> **Demo Workflow**: Present this notebook first to show how unstructured PDF documents are semantically chunked with `AI.CHUNK_DOCUMENT` and converted into structured graph relationships. Afterwards, transition to your **Cloud Spanner** and **BigQuery** SQL consoles to present the enterprise Property Graph queries and Reverse ETL pipeline across all product lines.

## 🏁 Phase 1: Environment Setup & API Configuration

### 🛠️ Verify & Resolve Python Client Libraries (`uv`)
Verify that `google-cloud-bigquery`, `google-cloud-storage`, `pandas`, and `db-dtypes` are importable. If running in an environment where any library is missing or broken, [`uv`](https://docs.astral.sh/uv/) dynamically resolves mutually compatible versions inside an isolated `.venv`—without hardcoded version pins or conflicts against Cloud Shell's pre-installed system packages. Because BigQuery's `AI.CHUNK_DOCUMENT` automatically provisions and manages the Document AI Layout Parser processor (`bq-created-layout-parser`), `google-cloud-documentai` is not required.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

REQUIRED_PACKAGES = [
    "google-cloud-bigquery",
    "google-cloud-storage",
    "pandas",
    "db-dtypes",
]

def _imports_healthy() -> bool:
    check_code = (
        "from google.cloud import bigquery, storage; "
        "import pandas, db_dtypes"
    )
    return subprocess.run(
        [sys.executable, "-c", check_code],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    ).returncode == 0

def _ensure_uv() -> str:
    local_bin = str(Path.home() / ".local" / "bin")
    path_env = f"{local_bin}:{os.environ.get('PATH', '')}"
    uv_bin = shutil.which("uv", path=path_env)
    if uv_bin:
        return uv_bin

    print("Installing uv...")
    subprocess.check_call(
        ["sh", "-c", "curl -LsSf https://astral.sh/uv/install.sh | sh"]
    )
    uv_bin = shutil.which("uv", path=path_env)
    if not uv_bin:
        raise RuntimeError("Failed to locate uv after installation.")
    return uv_bin

def ensure_clean_environment() -> None:
    if _imports_healthy():
        print("✅ Python client libraries verified and ready.")
        return

    uv_bin = _ensure_uv()
    in_virtualenv = sys.prefix != sys.base_prefix

    if in_virtualenv:
        print("Resolving and installing dependencies into active virtualenv with uv...")
        subprocess.check_call(
            [uv_bin, "pip", "install", "--python", sys.executable, *REQUIRED_PACKAGES]
        )
    else:
        # Isolate from Cloud Shell / system pre-installed packages (PEP 668 safe, no version pins)
        venv_dir = Path(".venv").resolve()
        venv_python = venv_dir / "bin" / "python"
        print("Creating isolated .venv with uv so system packages never conflict...")
        subprocess.check_call([uv_bin, "venv", "--python", sys.executable, "--clear", str(venv_dir)])
        subprocess.check_call(
            [uv_bin, "pip", "install", "--python", str(venv_python), *REQUIRED_PACKAGES]
        )
        site_pkgs = subprocess.check_output(
            [str(venv_python), "-c", "import sysconfig; print(sysconfig.get_paths()['purelib'])"],
            text=True,
        ).strip()
        plat_pkgs = subprocess.check_output(
            [str(venv_python), "-c", "import sysconfig; print(sysconfig.get_paths()['platlib'])"],
            text=True,
        ).strip()
        for pkg_dir in dict.fromkeys([plat_pkgs, site_pkgs]):
            if pkg_dir not in sys.path:
                sys.path.insert(0, pkg_dir)

    print("✅ Python client libraries resolved and installed cleanly via uv!")

ensure_clean_environment()

### ⚙️ Define Project Constants & Target Document
We detect your active Google Cloud project (`bq-dea` by default), configure the target `CHUNK_SIZE` (`250` tokens) for `AI.CHUNK_DOCUMENT`, and select **`gemini-3.8-flash`** for structured triple extraction on **`manual_frozone_zenith_5000.pdf`**.

In [ ]:
import subprocess

try:
    default_proj = subprocess.run(
        ["gcloud", "config", "get-value", "project"],
        capture_output=True,
        text=True,
        timeout=3,
        check=False,
    ).stdout.strip() or "bq-dea"
except Exception:
    default_proj = "bq-dea"

PROJECT_ID = default_proj  # @param {type:"string"}
LOCATION = "us"  # @param {type:"string"}
DATASET_ID = "kg_demo"  # @param {type:"string"}
GCS_BUCKET_NAME = f"{PROJECT_ID}-kg-demo-manuals"  # @param {type:"string"}
TARGET_DOCUMENT_URI = f"gs://{GCS_BUCKET_NAME}/manuals/manual_frozone_zenith_5000.pdf"  # @param {type:"string"}

OBJECT_TABLE_NAME = "frozone_manual_object_table"
PROCESSED_DOCUMENTS_TABLE = "frozone_parsed_chunks"
EXTRACTED_TRIPLES_TABLE = "frozone_extracted_triples"
CHUNK_SIZE = 250  # @param {type:"integer"}
GEMINI_MODEL_VERSION = "gemini-3.8-flash"  # @param {type:"string"}

print(f"✅ Active GCP Project        : {PROJECT_ID}")
print(f"✅ Target BigQuery Dataset   : {PROJECT_ID}.{DATASET_ID}")
print(f"✅ Target Engineering Manual : {TARGET_DOCUMENT_URI}")
print(f"✅ AI.CHUNK_DOCUMENT Size    : {CHUNK_SIZE} tokens")
print(f"✅ Target Gemini Model       : {GEMINI_MODEL_VERSION}")

### 🛠️ Enable Required APIs for `AI.CHUNK_DOCUMENT` & `AI.GENERATE`
To use `AI.CHUNK_DOCUMENT`, enable the Document AI API (`documentai.googleapis.com`). When `AI.CHUNK_DOCUMENT` runs, BigQuery automatically creates and reuses a managed Document AI Layout Parser processor named `bq-created-layout-parser` in your project—no manual processor or remote model creation required!

In [ ]:
!gcloud services enable documentai.googleapis.com aiplatform.googleapis.com bigqueryconnection.googleapis.com --project={PROJECT_ID} --quiet
print("✅ Document AI, Vertex AI, and BigQuery Connection APIs enabled.")

### 🛠️ Verify BigQuery Dataset & Stage Target PDF Document in GCS
Ensure `kg_demo` exists in BigQuery and verify that our target document (**`manual_frozone_zenith_5000.pdf`**) is staged in Google Cloud Storage.

In [ ]:
from pathlib import Path
from google.cloud import bigquery, storage

bq_client = bigquery.Client(project=PROJECT_ID)
dataset_ref = bq_client.dataset(DATASET_ID)
try:
    bq_client.get_dataset(dataset_ref)
    print(f"✅ BigQuery Dataset `{PROJECT_ID}.{DATASET_ID}` exists.")
except Exception:
    print(f"Creating BigQuery Dataset `{PROJECT_ID}.{DATASET_ID}`...")
    ds = bigquery.Dataset(dataset_ref)
    ds.location = LOCATION
    bq_client.create_dataset(ds, exists_ok=True)
    print(f"✅ Created BigQuery Dataset `{PROJECT_ID}.{DATASET_ID}`.")

storage_client = storage.Client(project=PROJECT_ID)
bucket = storage_client.bucket(GCS_BUCKET_NAME)
if not bucket.exists():
    bucket = storage_client.create_bucket(GCS_BUCKET_NAME, location=LOCATION)
    print(f"✅ Created GCS Bucket: gs://{GCS_BUCKET_NAME}/")
else:
    print(f"✅ GCS Bucket gs://{GCS_BUCKET_NAME}/ exists.")

# Stage manual_frozone_zenith_5000.pdf from local manuals/ (or ../manuals/) directory if present
for candidate in (
    Path("../manuals/manual_frozone_zenith_5000.pdf"),
    Path("manuals/manual_frozone_zenith_5000.pdf"),
    Path("sample_data/manual_frozone_zenith_5000.pdf"),
):
    if candidate.exists():
        blob = bucket.blob(f"manuals/{candidate.name}")
        if not blob.exists():
            blob.upload_from_filename(str(candidate))
            print(f"   • Uploaded {candidate.name} -> gs://{GCS_BUCKET_NAME}/manuals/{candidate.name}")
        else:
            print(f"   • Verified {candidate.name} is staged at gs://{GCS_BUCKET_NAME}/manuals/{candidate.name}")
        break

print("✅ Target Engineering Manual ready in Cloud Storage!")

## 📄 Phase 2: Document Ingestion & Semantic Chunking (`AI.CHUNK_DOCUMENT`)

BigQuery's table-valued **[`AI.CHUNK_DOCUMENT`](https://docs.cloud.google.com/bigquery/docs/reference/standard-sql/bigqueryml-syntax-ai-chunk-document)** function splits unstructured documents into semantically coherent text chunks based on document layout (headings, sections, paragraphs, lists, and tables). It supports three input sources:
1. **Cloud Storage URI literal** (e.g., `'gs://bucket/manuals/*.pdf'`) authenticated via End-User Credentials (EUC).
2. **External Object Table** (`TABLE project.dataset.object_table`) with a Cloud Resource Connection (`connection_id`).
3. **`ObjectRef` subquery** (e.g., `OBJ.LIST(...)` or `OBJ.MAKE_REF(...)`).

### 🔗 Configure External Object Table for `manual_frozone_zenith_5000.pdf`
First, we optionally configure a BigQuery External Object Table (`kg_demo.frozone_manual_object_table`) and grant its Cloud Resource Connection `roles/documentai.editor` (to create/invoke `bq-created-layout-parser`) and `roles/storage.objectViewer`.

In [ ]:
DEFAULT_CONNECTION_ID = f"{PROJECT_ID}.{LOCATION}.__default_cloudresource_connection__"

# Grant required IAM roles for AI.CHUNK_DOCUMENT (roles/documentai.editor & storage.objectViewer)
# and AI.GENERATE (roles/aiplatform.user) to the default BigQuery Cloud Resource Connection
sql_grant_connection = f"""
GRANT `roles/storage.objectViewer`, `roles/documentai.editor`, `roles/documentai.viewer`, `roles/aiplatform.user`
ON PROJECT `{PROJECT_ID}`
TO "connection:{DEFAULT_CONNECTION_ID}";
"""

sql_object_table = f"""
CREATE OR REPLACE EXTERNAL TABLE `{PROJECT_ID}.{DATASET_ID}.{OBJECT_TABLE_NAME}`
WITH CONNECTION `{DEFAULT_CONNECTION_ID}`
OPTIONS(
  object_metadata = 'SIMPLE',
  uris = ['{TARGET_DOCUMENT_URI}']
);
"""

try:
    bq_client.query(sql_object_table).result()
    print(f"✅ Object Table `{PROJECT_ID}.{DATASET_ID}.{OBJECT_TABLE_NAME}` created for `manual_frozone_zenith_5000.pdf`!")
    try:
        bq_client.query(sql_grant_connection).result()
        print("✅ IAM roles (Document AI Editor, Storage Object Viewer, Vertex AI User) granted to connection!")
    except Exception as e_grant:
        print(f"ℹ️ Connection IAM SQL grant note (EUC fallback remains available): {e_grant}")
    df_objects = bq_client.query(
        f"SELECT uri, content_type, size FROM `{PROJECT_ID}.{DATASET_ID}.{OBJECT_TABLE_NAME}`"
    ).to_dataframe()
    display(df_objects)
except Exception as e:
    print(f"ℹ️ Proceeding with direct Cloud Storage URI + EUC authentication ({e})")

### 📝 Semantically Chunk the Manual with `AI.CHUNK_DOCUMENT`
We run BigQuery SQL's **`AI.CHUNK_DOCUMENT`** with `chunk_size => 250` tokens to split the FroZone Zenith 5000 engineering manual along structural boundaries (`chunk_id`, `start_page`, `end_page`, `content`, `status`, `uri`) and store the chunks in `{DATASET_ID}.frozone_parsed_chunks`.

Notice how `AI.CHUNK_DOCUMENT`:
- Automatically provisions and reuses `bq-created-layout-parser` in Document AI without any `CREATE MODEL` DDL.
- Preserves Markdown section headers, schematic labels (`COMP-809`, `COIL-101`), BOM specifications, and troubleshooting advisories across pages!

In [ ]:
# Option A: AI.CHUNK_DOCUMENT directly over a Cloud Storage URI using End-User Credentials (EUC)
sql_chunk_uri = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.{PROCESSED_DOCUMENTS_TABLE}` AS
SELECT
  chunk_id,
  start_page,
  end_page,
  content,
  status,
  uri
FROM AI.CHUNK_DOCUMENT(
  '{TARGET_DOCUMENT_URI}',
  chunk_size => {CHUNK_SIZE}
)
WHERE status = '';
"""

# Option B: AI.CHUNK_DOCUMENT over an External Object Table with a Cloud Resource Connection
sql_chunk_object_table = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.{PROCESSED_DOCUMENTS_TABLE}` AS
SELECT
  chunk_id,
  start_page,
  end_page,
  content,
  status,
  uri
FROM AI.CHUNK_DOCUMENT(
  TABLE `{PROJECT_ID}.{DATASET_ID}.{OBJECT_TABLE_NAME}`,
  connection_id => '{DEFAULT_CONNECTION_ID}',
  chunk_size => {CHUNK_SIZE}
)
WHERE status = '';
"""

print(f"Executing AI.CHUNK_DOCUMENT (chunk_size => {CHUNK_SIZE}) on manual_frozone_zenith_5000.pdf...")
try:
    bq_client.query(sql_chunk_object_table).result()
    print(f"✅ Semantic chunks stored in `{PROJECT_ID}.{DATASET_ID}.{PROCESSED_DOCUMENTS_TABLE}` via AI.CHUNK_DOCUMENT (Object Table + Connection).")
except Exception as e_conn:
    print(f"ℹ️ Retrying AI.CHUNK_DOCUMENT directly on GCS URI with End-User Credentials ({e_conn})...")
    bq_client.query(sql_chunk_uri).result()
    print(f"✅ Semantic chunks stored in `{PROJECT_ID}.{DATASET_ID}.{PROCESSED_DOCUMENTS_TABLE}` via AI.CHUNK_DOCUMENT (GCS URI + EUC).")

# Preview semantic document chunks produced by AI.CHUNK_DOCUMENT
df_chunks = bq_client.query(
    f"""
    SELECT chunk_id, start_page, end_page, status, uri, content
    FROM `{PROJECT_ID}.{DATASET_ID}.{PROCESSED_DOCUMENTS_TABLE}`
    ORDER BY chunk_id
    """
).to_dataframe()
display(df_chunks)

## 💡 Phase 3: Knowledge Extraction with Gemini 3.8 Flash (`AI.GENERATE`)

We invoke **Gemini 3.8 Flash** (`AI.GENERATE` with `endpoint => 'gemini-3.8-flash'`) in BigQuery SQL across every semantic chunk returned by `AI.CHUNK_DOCUMENT`.
We pass a structured `output_schema` requesting an array of component relationships (`subject`, `subject_entity_type`, `relationship`, `object`, `object_entity_type`, `confidence`), `UNNEST` the array, and deduplicate across chunks into clean **Knowledge Graph Triples** stored in `{DATASET_ID}.frozone_extracted_triples`.

In [ ]:
sql_extract_kg = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.{EXTRACTED_TRIPLES_TABLE}` AS
WITH RawTriples AS (
  SELECT
    uri AS source_document,
    chunk_id,
    r.subject,
    r.subject_entity_type,
    r.relationship,
    r.object,
    r.object_entity_type,
    r.confidence
  FROM (
    SELECT
      uri,
      chunk_id,
      AI.GENERATE(
        '''
        You are a technical manufacturing knowledge graph extractor.
        Extract explicit component and material specifications from the engineering manual chunk.
        Identify the ENTITY TYPE for both subject and object.
        Valid Entity Types: Product, Part, Material.

        ### REQUIRED RELATIONSHIP TYPES (ONLY extract these two):
        * **CONTAINS_PART**: Product -> Part (e.g., 'FroZone Zenith 5000' CONTAINS_PART 'COMP-809' or 'Compressor Unit-XZ')
        * **MADE_OF**: Part -> Material (e.g., 'COMP-809' MADE_OF 'Fiberglass', 'Thermal Insulation Panel' MADE_OF 'Fiberglass')

        ### NORMALIZATION RULES:
        * Use canonical material names in Title Case: e.g., 'Fiberglass' (not 'Fiberglass Core' or 'fiberglass'), 'Freon-12', 'Copper Alloy C101', 'Stainless Steel 316'.
        * If a chunk has no component/material relationships, return an empty relationships array [].

        Document chunk:
        ''' || content,
        output_schema => '''
          relationships ARRAY<STRUCT<
            subject STRING,
            subject_entity_type STRING,
            relationship STRING,
            object STRING,
            object_entity_type STRING,
            confidence FLOAT64
          >>
        ''',
        endpoint => '{GEMINI_MODEL_VERSION}'
      ) AS extracted_data
    FROM `{PROJECT_ID}.{DATASET_ID}.{PROCESSED_DOCUMENTS_TABLE}`
    WHERE status = ''
  ),
  UNNEST(extracted_data.relationships) AS r
  WHERE r.subject IS NOT NULL
    AND r.object IS NOT NULL
    AND r.relationship IN ('CONTAINS_PART', 'MADE_OF')
)
SELECT
  ANY_VALUE(source_document) AS source_document,
  MIN(chunk_id) AS first_chunk_id,
  subject,
  ANY_VALUE(subject_entity_type) AS subject_entity_type,
  relationship,
  object,
  ANY_VALUE(object_entity_type) AS object_entity_type,
  MAX(confidence) AS confidence
FROM RawTriples
GROUP BY subject, relationship, object;
"""

print(f"Executing AI.GENERATE with {GEMINI_MODEL_VERSION} over semantic chunks from AI.CHUNK_DOCUMENT...")
bq_client.query(sql_extract_kg).result()
print(f"✅ Extracted KG triples stored in `{PROJECT_ID}.{DATASET_ID}.{EXTRACTED_TRIPLES_TABLE}`.")

# Ensure the enterprise extracted_knowledge_graph table exists and merge new triples
sql_ensure_enterprise_kg = f"""
CREATE TABLE IF NOT EXISTS `{PROJECT_ID}.{DATASET_ID}.extracted_knowledge_graph` (
  subject STRING,
  subject_entity_type STRING,
  relationship STRING,
  object STRING,
  object_entity_type STRING,
  confidence FLOAT64,
  source_document STRING
);
"""
sql_merge_triples = f"""
INSERT INTO `{PROJECT_ID}.{DATASET_ID}.extracted_knowledge_graph`
  (subject, subject_entity_type, relationship, object, object_entity_type, confidence, source_document)
SELECT
  subject, subject_entity_type, relationship, object, object_entity_type, confidence, source_document
FROM `{PROJECT_ID}.{DATASET_ID}.{EXTRACTED_TRIPLES_TABLE}` AS new_triples
WHERE NOT EXISTS (
  SELECT 1
  FROM `{PROJECT_ID}.{DATASET_ID}.extracted_knowledge_graph` AS existing
  WHERE existing.subject = new_triples.subject
    AND existing.relationship = new_triples.relationship
    AND existing.object = new_triples.object
);
"""
bq_client.query(sql_ensure_enterprise_kg).result()
bq_client.query(sql_merge_triples).result()
print(f"✅ Merged extracted triples into `{PROJECT_ID}.{DATASET_ID}.extracted_knowledge_graph`.")

df_triples = bq_client.query(
    f"""
    SELECT first_chunk_id, subject, subject_entity_type, relationship, object, object_entity_type, confidence
    FROM `{PROJECT_ID}.{DATASET_ID}.{EXTRACTED_TRIPLES_TABLE}`
    ORDER BY first_chunk_id, relationship, subject
    """
).to_dataframe()
display(df_triples)

## 🔍 Phase 4: Inspecting the Extracted Graph Structure for `FroZone Zenith 5000`
Let's inspect the multi-hop component tree extracted from **`manual_frozone_zenith_5000.pdf`**:
1. **`FroZone Zenith 5000`** (`Product`) -> **`CONTAINS_PART`** -> **`COMP-809`** / **`Thermal Insulation Panel`** (`Part`) -> **`MADE_OF`** -> **`Fiberglass`** (`Material`)
2. **`FroZone Zenith 5000`** (`Product`) -> **`CONTAINS_PART`** -> **`Compressor Unit-XZ`** (`Part`) -> **`MADE_OF`** -> **`Freon-12`** (`Material`)
3. **`FroZone Zenith 5000`** (`Product`) -> **`CONTAINS_PART`** -> **`COIL-101`** (`Part`) -> **`MADE_OF`** -> **`Copper Alloy C101`** (`Material`)

Notice how `AI.CHUNK_DOCUMENT` + **Gemini 3.8 Flash** (`AI.GENERATE`) transformed unstructured PDF pages into a clean, multi-hop component relationship tree!

In [ ]:
sql_preview_tree = f"""
SELECT
  t1.subject AS root_product,
  t1.relationship AS rel_1,
  t1.object AS component_part,
  t2.relationship AS rel_2,
  t2.object AS material,
  CASE
    WHEN t2.object IN ('Fiberglass', 'Freon-12') THEN '⚠️ HAZARDOUS MATERIAL DETECTED'
    ELSE 'NORMAL'
  END AS safety_flag
FROM `{PROJECT_ID}.{DATASET_ID}.{EXTRACTED_TRIPLES_TABLE}` t1
JOIN `{PROJECT_ID}.{DATASET_ID}.{EXTRACTED_TRIPLES_TABLE}` t2
  ON t1.object = t2.subject
WHERE t1.relationship = 'CONTAINS_PART' AND t2.relationship = 'MADE_OF'
ORDER BY safety_flag DESC, component_part, material;
"""
df_tree = bq_client.query(sql_preview_tree).to_dataframe()
print("🏗️ Extracted Multi-Hop Bill of Materials Tree for FroZone Zenith 5000:")
display(df_tree)

## 🎉 Next Step: Building & Querying the Complete Enterprise Graph across All Product Lines!

Now that you have demonstrated how an unstructured engineering manual (**`manual_frozone_zenith_5000.pdf`**) is semantically chunked by **`AI.CHUNK_DOCUMENT`** and transformed into structured Knowledge Graph Triples by **Gemini 3.8 Flash** (`AI.GENERATE`):

### 🚀 Check the scripts
As a pre-requisite, `sql/00_setup_bigquery_tables.sql` uses `AI.CHUNK_DOCUMENT('gs://.../manuals/*.pdf')` to chunk and extract triples across **all four product lines** (`FroZone Zenith 5000`, `Arctic Swirl 5000`, `Blizzard Pro 300`, and `FrostGuard Commercial 2000`).

Switch to your SQL consoles to see the components end-to-end:
1. **Part 1 (`sql/01_spanner_transactional_graph.sql`)** in **Cloud Spanner Studio**: Explore the operational `TransactionalGraph` showing customer orders and product catalogs.
2. **Part 2 (`sql/02_bigquery_knowledge_graph.sql`)** in **BigQuery Studio**: Define the enterprise `CREATE PROPERTY GRAPH kg_demo.manufacturing_kg` over all extracted manuals and run multi-hop GQL and GraphRAG (`VECTOR_SEARCH` + `GRAPH_TABLE`) queries.
3. **Part 3 (`sql/03_bq_generate_insights_reverse_etl.sql`)** in **BigQuery Studio**: Generate AI Product Advisories and Customer Risk profiles from the GQL hazard findings using `gemini-3.8-flash`.
4. **Reverse ETL Sync**: Run `uv --directory scripts run reverse_etl_sync.py --project={PROJECT_ID}` (or `sql/03_bq_native_reverse_etl_export.sql` on Enterprise Edition) to sync BigQuery AI advisories into Cloud Spanner.
5. **Part 4 (`sql/04_spanner_operational_insights_graph.sql`)** in **Cloud Spanner Studio**: Query the enriched `OperationalInsightsGraph` to see customer service reps access live customer orders joined with AI Graph advisories in real time!

### Thinking of doing this in production?

Ask the Data Engineering Agent to create a pipeline for you. Install the [Data Agent Kit]() Here's a prompt you can use:

```text
Create a data engineering pipeline to independently and idempotently execute the diferent steps in the notebook kg_demo_manufacturing every n hours.
The goal is to keep the knowledge graph in BigQuery updated and keep the reverse ETL continuous query executing. Use dbt and managed apache airflow. Consider development, staging and production environments with different project IDs, where the development environment has sample data you need to seed in a developer-specific temporary dataset.
```